# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Angel Fernandez Garcia  <br>
Url: https://github.com/.../03MAIR---Algoritmos-de-Optimizacion---/tree/master/TrabajoPractico<br>
Google Colab: https://colab.research.google.com/drive/xxxxxxxxxxxxxxxx <br>
Problema:
>1. Sesiones de doblaje <br>

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en
las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el
estudio de grabación independientemente del número de tomas que se graben. No es
posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de
manera que el gasto por los servicios de los actores de doblaje sea el menor posible.

- Numero de actores: 30
- Numero de tomas: 10                

#Ideas para implementar:

    - Porgramacion lineal entera (ILP / MILP)
    - Recocido simulado y busqueda tabu

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

#Respuesta

Es espacio de soluciones se presenta mediante un vector de 30 valores, donde cada valor i representa en dia en el que se debe grabar la toma i. De manera que si el primer valor del vector solucion es 4, significa que la toma 1 se debe grabar el cuarto dia.

$$ S = (t_1, t_2, ..., t_{30})$$

La funcion objetivo suma para 1 cada dia por cada actor diferente que haya participado entre todas las tomas, se consigue el resultado de forma acumulativa por cada dia.

$$ \sum_{\text{días}} | \text{Actores distintos ese día} | $$

Se han implementado dos funciones de restriccion para este problema. La primera, que es mas evidente, verifica que para ningún dia se hayan establecido mas de 6 tomas. La segunda, nace de un problema que podríamos enfrentarnos en la busqueda de una solucion, verifica que no haya dias vacios entre medias. No querríamos llegar a una solucion que nos pidiera que dejasemos el dia 3 sin tomas, es una situación que la  función objetivo no contempla.

#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

#Respuesta

Este problema se trata de un problema NP-Dificil, que guarda similitud con el problema de Set Partitioning. El problema Set Partitioning consiste en didivir un conjunto de elementos en subconjuntos de manera que se cumplas ciertas condiciones de cobertura.

Nuestro problema de organizacion de tomas en dias se puede transformar en tiempo polinomico a el problema de Set partitioning, entendiendo las tomas como elementos y las dias como subconjuntos, que deben cumplir la condicion de ser menos de 7 elementos por subconjunto. Por último, cada elemento solo puede y debe pertenecer a un subconjunto y se debe particionar de forma que minimicen una función objetivo.

El espacio de soluciones exacto es dificil de calcular ya que dependemos de la cantidad maxima de dias de los que disponemos, si tomamos de forma aproximada 15 (ya que entendiendo el problema no se cree que se vayan a necesitar tantos) como cantidad maxima. Estos son los principales puntos a tener en cuenta:

- 30 tomas (el orden importa)
- cada toma asignada a un dia
- maximo 6 tomas por dia
- minimo 1 toma por dia
- maximo 15 dias

$$ \sum_{d=1}^{15} \left( \sum_{\substack{n_1 + \dots + n_d = 30 \\ 1 \le n_i \le 6}} \frac{30!}{n_1! n_2! \dots n_d!} \right) \approx 10^{22} $$


Sin restricciones de la cantidad de tomas por dia, el numero de soluciones seria $15^{30}$, con restricciones baja mucho pero el numero sigue siendo monstruosamente grande como para emplear un algoritmo exacto que explorase todas las soluciones.

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

#Respuesta

Dada la enorme cantidad de soluciones se ha optado por emplear tecnicas metaheuristicas que evitan explorar todas las soluciones y se aproximan a minimos locales de forma inteligente intentando encontrar soluciones globales. En concreto, se ha optado por emplear un algoritmo genetico dado que se trata de un problema cuyos datos son faciles de mutar y cruzar, además de ser un algoritmo altamente potente, que en clase solamente hemos visto en teoria y no en práctica, por lo que el interes de ponerlo a prueba era mayor.

Otras soluciones para este problema seria considerando otras metaheuristicas como Simmulated Annealing o Tabu Search.

## Carga de datos y librerias

In [1]:
import pandas as pd
import numpy as np
import random

In [2]:
ruta_datos = "Datos problema doblaje(30 tomas, 10 actores) - Hoja 1.csv"

datos = pd.read_csv(ruta_datos, header=1)

datos.head()

,Toma,1,2,3,4,5,6,7,8,9,10,Unnamed: 11,Total
0,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,5.0
1,2,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,3.0
2,3,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,NaN,3.0
3,4,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,NaN,4.0
4,5,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,3.0


## Funciones de apoyo

In [3]:
def menos_de_6_tomas_por_dia(solucion):
    """
    Funcion booleana que verifica si la solucion contiene algun dia con mas de 6 tomas
    """
    _, recuento = np.unique(solucion, return_counts=True)
    # print(recuento)

    return np.max(recuento) < 7

# valores_aleatorios = np.random.randint(7, size=30)
# print(valores_aleatorios)
# menos_de_6_tomas_por_dia(valores_aleatorios)

In [4]:
def no_hay_dias_intermedios_vacios(solucion):
    """
    Funcion booleana que verifica que no existan dias intermedios sin tomas
    """
    set_solucion_ordenado = set(sorted(solucion)) #sorted convierte la estructura a una lista
    # print(set_solucion_ordenado)
    set_completo = set(range(0, solucion.max()+1))
    # print(set_completo)

    return set_solucion_ordenado == set_completo

# valores_aleatorios = np.random.randint(10, size=30)
# print(valores_aleatorios)
# no_hay_dias_intermedios_vacios(valores_aleatorios)

In [5]:
def validar_poblacion(poblacion):
    """
    Devuelve True si todos los individuos de una poblacion cumplen con las restricciones
    """

    for individuo in poblacion:
        if (no_hay_dias_intermedios_vacios(individuo) == False) or (menos_de_6_tomas_por_dia(individuo) == False):
            return False

    return True

In [6]:
def funcion_objetivo(solucion, datos):
    """
    Recibe una lista de 30 elementos y los datos donde comparar los actores que trabajar en cada toma
    El elemento 0 mantiene el valor de la toma 1 y el numero que contiene es el dia en el que esta se graba
    Devuelve un valor entero
    """

    datos_del_dia = None
    tomas_en_el_dia = None
    actores_a_pagar = 0

    for dia in range(0, solucion.max()+1):
        # print(f'Dia: {dia}')
        datos_del_dia = np.zeros((6, 10))
        tomas_en_el_dia = 0

        for toma in range(len(solucion)):
            if solucion[toma] == dia:
                # print(f'Tomas en el dia: {tomas_en_el_dia}')
                datos_del_dia[tomas_en_el_dia] = datos.iloc[toma][1:11]
                tomas_en_el_dia += 1

        actores_a_pagar += np.any(datos_del_dia, axis=0).sum()
        # print(actores_a_pagar)

    return actores_a_pagar

# valores_aleatorios = np.random.randint(10, size=30)
# print(valores_aleatorios)
# if menos_de_6_tomas_por_dia(valores_aleatorios):
#     print(f'Resultado funcion objetivo: {funcion_objetivo(valores_aleatorios, datos)}')
# else :
#     print("Mas de 6 tomas en el dia")

In [7]:
def crear_solucion_aleatoria():
    return np.random.randint(20, size=30)

def generar_poblacion_aleatoria():
    """
    Crea una poblacion de 60 individios generados de forma aleatoria
    Devuelve una lista de 60 soluciones aleatorias
    """

    return [crear_solucion_aleatoria() for _ in range(60)]

# poblacion = generar_poblacion_aleatoria()
# len(poblacion)
# padres = random.sample(poblacion, 2)

In [8]:
def torneo(poblacion, datos, k=3):
    """
    Se escoge un subconjunto aleatorios de la poblacion de k tamaño y se evaluan, entonces, se escoge el mejor
    """
    padres = random.sample(poblacion, k)
    ganador = padres[0]
    mejor_resultado = funcion_objetivo(padres[0], datos)

    for padre in padres[1:]:
        resultado_padre = funcion_objetivo(padre, datos)

        if resultado_padre < mejor_resultado: # Se busca  minimizar el resultado
            ganador = padre
            mejor_resultado = resultado_padre

    return ganador


def seleccionar_padres(poblacion, datos, k=3):
    """
    Se realiza una seleccion de padres mediante torneo
    Se realiza 60 veces con el objetivo de obtener 60 padres
    Se devuelve una lista de 60 individuos
    """

    return [torneo(poblacion, datos, k) for _ in range(60)]

# resultado = seleccionar_padres(generar_poblacion_aleatoria(), datos)
# len(resultado)
# funcion_objetivo(resultado[0], datos)

In [9]:
def crossover(padreA, padreB):
    """
    Funcion de crossover de punto simple
    Recibe 2 padres
    Calcula un numero aleatorio del 0 al 29 y corta en ese punto
    Devuelve 2 hijos
    """

    punto = np.random.randint(0, len(padreA))

    hijo1 = np.concatenate((padreA[:punto], padreB[punto:]), axis=None)
    hijo2 = np.concatenate((padreB[:punto], padreA[punto:]), axis=None)

    return hijo1, hijo2

# crossover(resultado[0], resultado[1])

def crossover_poblacion(poblacion):
    """
    Recibe una poblacion de 60 padres
    Devuelve un poblacion de 60 hijos
    """
    resultado = poblacion.copy() # Se hace una copia aunque no se espera mantener ninguno de los objetidos originales
    np.random.shuffle(resultado)

    for i in range(30):
        hijoA, hijoB = crossover(poblacion[i], poblacion[i+30])
        resultado[i] = hijoA
        resultado[i+30] = hijoB

    return resultado

# crossover_poblacion(generar_poblacion_aleatoria())

In [10]:
def mutar(individuo):
    """
    Devuelve el individuo con 5 valores mutados
    """

    resultado = individuo.copy()
    tomas_a_cambiar = np.random.randint(30, size=1)

    for toma in tomas_a_cambiar:
        nueva_toma = np.random.randint(low=0, high=max(individuo)+1) # max + 1 nos aseguramo que la cantidad de dias
                                                                    # no puede explotar de forma aleatoria y romper la restriccion
        resultado[toma] = nueva_toma

    return resultado

def mutar_poblacion(poblacion, p=0.3):
    """
    Cada hijo de la poblacion tiene una probabilidad p de mutar
    Devuelve la poblacion con algunos individuos mutados
    """

    resultado = np.copy(poblacion)

    for i in range(len(poblacion)):
        if np.random.rand() <= p:
            resultado[i] = mutar(poblacion[i])

    return resultado

# mutar_poblacion(resultado)

In [11]:
def reparar(poblacion):
    """
    En caso de que todos los individuos de la poblacion cumplan los requisitos, devuelve la poblacion original
    En caso contrario, se reparan los individuos que no los cumplan
    Mientras no cumpla tener maximo 6 tomas por dia, cambia la primera toma del dia mas ocupado al menos
    Mientras no cumpla no dejar dias intermedios vacios, adelanta un dia tomas las tomas de dias posteriores al vacio
    """

    resultado = poblacion.copy()

    for i in range(len(resultado)):
        while menos_de_6_tomas_por_dia(resultado[i]) == False:
            # print(f'mas de 6 tomas en algun dia: {poblacion[i]}')
            dia, recuento = np.unique(resultado[i], return_counts=True)
            toma_a_cambiar = dia[np.argmax(recuento)]
            for j in range(len(resultado[i])):
                if resultado[i][j] == toma_a_cambiar:
                    resultado[i][j] = dia[np.argmin(recuento)]
                    break

        while no_hay_dias_intermedios_vacios(resultado[i]) == False:
            # print(f'dias intermedios: {poblacion[i]}')
            for j in range(21):
                if j not in resultado[i]:
                    resultado[i][resultado[i] > j] = resultado[i][resultado[i] > j] - 1

    return resultado

# listaA = np.array([0, 1, 2, 3, 4, 5, 6, 0, 0, 0, 1, 1, 2, 2, 3, 4])
# listaB = np.array([0, 1, 2, 3, 4, 5, 6, 0, 0, 0, 1, 1, 2, 2, 3, 4, 8])

# print(listaB)
# np.argmin(listaB)
# listaB[listaB > 5] = listaB[listaB > 5] - 1
# print(listaB)
# reparar(resultado)

In [12]:
def reemplazo(padres, hijos, datos):
    """
    La funcion de reemplazo se encarga de mantener los 40 mejores padres y sustituir los otros 20 por los 20 mejores hijos
    """
    padres_scores = [funcion_objetivo(padre, datos) for padre in padres]
    padres_scores_ordenados = np.argsort(padres_scores) # Se queda con los indices de los mejores
    mejores_padres = [padres[i] for i in padres_scores_ordenados[:40]]

    hijos_scores = [funcion_objetivo(hijo, datos) for hijo in hijos]
    hijos_scores_ordenados = np.argsort(hijos_scores)
    mejores_hijos = [hijos[i] for i in hijos_scores_ordenados[:20]]

    resultado = mejores_padres + mejores_hijos

    return resultado

# resultado = reemplazo(reparar(generar_poblacion_aleatoria()), reparar(crossover_poblacion(generar_poblacion_aleatoria())), datos)
# resultado

## Algoritmo principal

In [29]:
def algoritmo_genetico(datos):
    """
    Busca mediante un algoritmo genetico la mejor solucion para nuestros datos
    Devuelve la mejor solucion junto a su score
    """

    poblacion = generar_poblacion_aleatoria()
    poblacion = reparar(poblacion)

    iteracion = 0

    mejor_resultado = poblacion[0]
    mejor_score = funcion_objetivo(poblacion[0], datos)

    while iteracion < 600:
        print(f'{iteracion}: {mejor_score}')

        padres_seleccionados = seleccionar_padres(poblacion, datos)

        hijos_creados = crossover_poblacion(padres_seleccionados)

        hijos_mutados = mutar_poblacion(hijos_creados)

        hijos_seguros = reparar(hijos_mutados)

        poblacion = reemplazo(padres_seleccionados, hijos_seguros, datos)

        resultados_poblacion = [funcion_objetivo(individuo, datos) for individuo in poblacion]

        resultados_poblacion_ordenado = np.sort(resultados_poblacion)
        # print(resultados_poblacion_ordenado)

        if resultados_poblacion_ordenado[0] < mejor_score:
            mejor_score = resultados_poblacion_ordenado[0]
            mejor_resultado = poblacion[np.argsort(resultados_poblacion)[0]]

        iteracion += 1

    return mejor_resultado, mejor_score

resultado, puntuacion = algoritmo_genetico(datos)
print(f'La mejor puntiacion encontrada ha sido: {puntuacion}')
print(f'El mejor resultado ha sido: {resultado}')

0: 67
1: 57
2: 55
3: 53
4: 52
5: 51
6: 49
7: 49
8: 49
9: 48
10: 48
11: 47
12: 47
13: 45
14: 45
15: 42
16: 42
17: 42
18: 42
19: 40
20: 40
21: 40
22: 39
23: 38
24: 38
25: 38
26: 38
27: 37
28: 37
29: 37
30: 37
31: 35
32: 35
33: 35
34: 34
35: 34
36: 34
37: 34
38: 34
39: 34
40: 34
41: 34
42: 34
43: 34
44: 34
45: 34
46: 34
47: 34
48: 34
49: 34
50: 34
51: 34
52: 34
53: 33
54: 33
55: 33
56: 33
57: 33
58: 33
59: 33
60: 33
61: 33
62: 33
63: 33
64: 33
65: 33
66: 33
67: 33
68: 33
69: 33
70: 33
71: 33
72: 33
73: 33
74: 33
75: 33
76: 33
77: 33
78: 33
79: 33
80: 33
81: 33
82: 33
83: 33
84: 33
85: 33
86: 33
87: 33
88: 33
89: 33
90: 33
91: 33
92: 33
93: 33
94: 33
95: 33
96: 33
97: 33
98: 33
99: 33
100: 33
101: 33
102: 33
103: 33
104: 33
105: 33
106: 33
107: 33
108: 33
109: 33
110: 33
111: 33
112: 33
113: 33
114: 33
115: 33
116: 33
117: 33
118: 33
119: 33
120: 33
121: 33
122: 33
123: 32
124: 32
125: 32
126: 32
127: 32
128: 32
129: 32
130: 32
131: 32
132: 32
133: 32
134: 32
135: 32
136: 32
137: 32
138: 3